<a href="https://colab.research.google.com/github/iv-Alena/credit-default-project/blob/main/model_UCI_Credit_Cardipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import os
import joblib

In [ ]:
#загружаем данные, удалям индексный столбец, выделяем целевой признак
df = pd.read_csv('../data/UCI_Credit_Card.csv')
df = df.drop(columns=['ID'])
X = df.drop('default.payment.next.month', axis=1)
y = df['default.payment.next.month']
# делим на обучеющию и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# создаем модель
model = Pipeline(steps=[('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000, random_state=42))])
# обучаем модель
model.fit(X_train, y_train)
# предсказания
y_pred = model.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))

Accuracy: 0.8076666666666666
Precision: 0.6868250539956804
Recall: 0.23963828183873398
F1-score: 0.3553072625698324


In [ ]:
# Сохраняем обученную модель
joblib.dump(model, "model.pkl")

['model.pkl']

In [ ]:
#загрузка модели и предсказания
model_predict = '''import joblib
import pandas as pd

def load_model(model_path="models/model.pkl"):
  # Загружает обученную модель из файла.
  model = joblib.load(model_path)
  return model

def make_prediction(data, model_path="models/model.pkl"):
  # Делает предсказание по данным клиента.

  model = load_model(model_path)

  if isinstance(data, dict):
    data = pd.DataFrame([data])

  prediction = model.predict(data)
  prediction_proba = model.predict_proba(data)

  return {"prediction": int(prediction[0]),
          "probability_no_default": float(prediction_proba[0][0]),
          "probability_default": float(prediction_proba[0][1])}
'''

with open("predict.py", "w", encoding="utf-8") as file:
 file.write(model_predict)



In [ ]:
# сохраняем apр
app = '''from flask import Flask, request, jsonify
import joblib
import pandas as pd


app = Flask(__name__)

MODEL_PATH = "models/model.pkl"

model = joblib.load(MODEL_PATH)


@app.route("/health", methods=["GET"])
def health():
   return jsonify({"status": "ok"})


@app.route("/predict", methods=["POST"])
def predict():
  try:
    data = request.get_json()

    if data is None:
     return jsonify({"error": "Данные не переданы"}), 400

    input_data = pd.DataFrame([data])

    prediction = model.predict(input_data)[0]
    prediction_proba = model.predict_proba(input_data)[0]

    result = {"prediction": int(prediction),
              "prediction_text": "default" if prediction == 1 else "no_default",
              "probability_no_default": float(prediction_proba[0]),
              "probability_default": float(prediction_proba[1])}

    return jsonify(result)

  except Exception as e:
    return jsonify({"error": str(e)}), 500


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)
'''

with open("app.py", "w", encoding="utf-8") as file:
    file.write(app)